# Шаг 05 — агентный цикл руками

Одного раунда мало: модель хочет сначала осмотреться (`list_files`), потом прочитать нужное (`read_file`). Крутим цикл, пока модель не ответит без `tool_calls`:

```
модель → tool_calls? → да  → выполнить, добавить результаты → модель → …
                     → нет → это финальный ответ
```

Этот цикл **и есть агент**. Инструменты теперь лежат в `tools.py`.

In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  # reads OPENROUTER_API_KEY and MODEL from .env

llm = ChatOpenAI(
    model=os.getenv("MODEL", "openai/gpt-4o-mini"),
    base_url="https://openrouter.ai/api/v1",   # OpenRouter speaks the OpenAI protocol
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [ ]:
from tools import list_files, read_file

TOOLS = [list_files, read_file]
for t in TOOLS:
    print(f"{t.name}: {t.description}")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

llm_with_tools = llm.bind_tools(TOOLS)
tools_by_name = {t.name: t for t in TOOLS}


def run_agent(messages: list) -> str:
    """Loop until the model answers without asking for tools."""
    for _ in range(10):                      # safety net against endless loops
        ai = llm_with_tools.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai.content                # no tools requested → final answer
        for call in ai.tool_calls:
            print(f"  ⚙ {call['name']}({call['args']})")
            result = tools_by_name[call["name"]].invoke(call["args"])
            messages.append(ToolMessage(result, tool_call_id=call["id"]))
    return "(stopped: too many steps)"

Добавим чат из шага 02 — агент помнит предыдущие вопросы.

In [ ]:
SYSTEM = ("Ты ассистент по коду. Смотри реальные файлы через инструменты, "
          "ничего не выдумывай. Отвечай кратко.")

history = [SystemMessage(SYSTEM)]


def ask(text: str) -> None:
    history.append(HumanMessage(text))
    print("ai ›", run_agent(history))

In [ ]:
ask("Какие файлы лежат в текущей папке и что делает tools.py? Кратко.")

In [ ]:
ask("Какие из этих инструментов могут что-то изменить на диске?")

Сколько сообщений накопилось за два вопроса — каждый вызов инструмента это пара AIMessage(tool_calls) + ToolMessage:

In [ ]:
from collections import Counter

Counter(type(m).__name__ for m in history)